In [1]:
# Import libraries and define configurations
import json
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
import joblib
from sklearn import set_config
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

RANDOM_STATE = 121
N_JOBS = -1

np.random.seed(RANDOM_STATE)


CURRENT_WORKING_DIRECTORY = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_WORKING_DIRECTORY.parent
    if CURRENT_WORKING_DIRECTORY.name == "notebooks"
    else CURRENT_WORKING_DIRECTORY
)

PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "data" / "processed"
REPORTS_DIRECTORY = PROJECT_ROOT / "reports"
MODELS_DIRECTORY = PROJECT_ROOT / "models"

ENGINEERED_FEATURES_PARQUET_PATH = (
    PROCESSED_DATA_DIRECTORY / "donor_features.parquet"
)
ENGINEERED_FEATURES_CSV_PATH = (
    PROCESSED_DATA_DIRECTORY / "donor_features.csv"
)
CLEANED_DONOR_DATA_PATH = (
    PROCESSED_DATA_DIRECTORY / "cleaned_donor_data.csv"
)
FEATURE_DICTIONARY_PATH = (
    REPORTS_DIRECTORY / "feature_dictionary.csv"
)

MODEL_PREDICTIONS_PATH = (
    PROCESSED_DATA_DIRECTORY / "model_predictions.csv"
)
FINAL_PRIMARY_PIPELINE_PATH = (
    MODELS_DIRECTORY / "final_primary_donor_pipeline.joblib"
)
MODEL_COMPARISON_RESULTS_PATH = (
    REPORTS_DIRECTORY / "05_model_comparison_results.csv"
)
CLASSIFICATION_MODELING_REPORT_PATH = (
    REPORTS_DIRECTORY / "05_classification_modeling.md"
)

MODELS_DIRECTORY.mkdir(parents=True, exist_ok=True)
REPORTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sns.set_theme(style="whitegrid")
set_config(display="diagram")

warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# Load the engineered dataset and feature dictionary
def format_project_path(path):
    relative_path = Path(path).resolve().relative_to(PROJECT_ROOT)
    return f"/{PROJECT_ROOT.name}/{relative_path.as_posix()}"


artifact_availability = pd.DataFrame({
    "artifact": [
        "Engineered features Parquet",
        "Engineered features CSV fallback",
        "Feature dictionary",
        "Cleaned donor dataset",
    ],
    "path_object": [
        ENGINEERED_FEATURES_PARQUET_PATH,
        ENGINEERED_FEATURES_CSV_PATH,
        FEATURE_DICTIONARY_PATH,
        CLEANED_DONOR_DATA_PATH,
    ],
})

artifact_availability["path"] = artifact_availability["path_object"].apply(
    format_project_path
)
artifact_availability["available"] = artifact_availability["path_object"].apply(
    Path.exists
)

display(artifact_availability[
    ["artifact", "path", "available"]
].style
    .hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
)

if ENGINEERED_FEATURES_PARQUET_PATH.exists():
    modeling_data = pd.read_parquet(
        ENGINEERED_FEATURES_PARQUET_PATH
    )
    modeling_data_source = "Parquet"
    modeling_data_path = ENGINEERED_FEATURES_PARQUET_PATH

elif ENGINEERED_FEATURES_CSV_PATH.exists():
    modeling_data = pd.read_csv(
        ENGINEERED_FEATURES_CSV_PATH
    )
    modeling_data_source = "CSV fallback"
    modeling_data_path = ENGINEERED_FEATURES_CSV_PATH

else:
    raise FileNotFoundError(
        "Neither donor_features.parquet nor donor_features.csv "
        "was found in the processed data directory."
    )

if not FEATURE_DICTIONARY_PATH.exists():
    raise FileNotFoundError(
        f"Feature dictionary not found: "
        f"{format_project_path(FEATURE_DICTIONARY_PATH)}"
    )

feature_dictionary = pd.read_csv(
    FEATURE_DICTIONARY_PATH
)

print(f"\nModeling dataset source: {modeling_data_source}")
print(
    "Modeling dataset path:",
    format_project_path(modeling_data_path),
)
print(
    "Feature dictionary path:",
    format_project_path(FEATURE_DICTIONARY_PATH),
)

print(
    "\nModeling dataset loaded:",
    f"{modeling_data.shape[0]:,} rows × "
    f"{modeling_data.shape[1]:,} columns",
)

print(
    "Feature dictionary loaded:",
    f"{feature_dictionary.shape[0]:,} rows × "
    f"{feature_dictionary.shape[1]:,} columns",
)

artifact,path,available
Engineered features Parquet,/red-cross-donor-prediction/data/processed/donor_features.parquet,True
Engineered features CSV fallback,/red-cross-donor-prediction/data/processed/donor_features.csv,True
Feature dictionary,/red-cross-donor-prediction/reports/feature_dictionary.csv,True
Cleaned donor dataset,/red-cross-donor-prediction/data/processed/cleaned_donor_data.csv,True



Modeling dataset source: Parquet
Modeling dataset path: /red-cross-donor-prediction/data/processed/donor_features.parquet
Feature dictionary path: /red-cross-donor-prediction/reports/feature_dictionary.csv

Modeling dataset loaded: 34,403 rows × 55 columns
Feature dictionary loaded: 77 rows × 7 columns
